# 잠긴 종 룰북 검색 실험

룰북에서 질문에 답하는 근거를 찾습니다. **의미 검색**과 **의미에 키워드도 더한 검색**을 같은 10문항으로 비교합니다. 마지막에는 Qwen이 고른 룰북 문장과 쪽수를 확인합니다.

## 실행 방법

1. Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다. 검색 결과만 확인할 때는 CPU로도 실행할 수 있습니다.
2. **모두 실행**을 누릅니다.
3. 파일 선택창이 뜨면 **룰북 PDF를 업로드합니다.**

**실험에 사용한 룰북**  
[D&D Basic Rules v1.0 한국어판 PDF](https://github.com/TRPG-masterbot/data/blob/main/Rules.pdf) (179쪽). 아래 결과를 재현하려면 같은 PDF를 사용합니다. 파일 이름은 자유롭게 지정할 수 있습니다.


In [1]:
# 벡터 저장 위치. 기본값으로 실행해도 됩니다.
USE_DRIVE_CACHE = False  # 새 Colab 세션에서도 벡터를 다시 쓰려면 True

from pathlib import Path

cache_dir = Path("/content/rule_cache")
if USE_DRIVE_CACHE:
    from google.colab import drive

    drive.mount("/content/drive")
    cache_dir = Path("/content/drive/MyDrive/잠긴종_RAG_캐시")


In [2]:
# 1. PDF 읽기, 검색, 답변 생성에 필요한 패키지 설치
%pip -q install "pymupdf>=1.24,<2" "sentence-transformers>=5,<6" "chromadb>=1,<2" "transformers>=5.3,<6" "accelerate>=1,<2" "rank-bm25==0.2.2"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 require

## 룰북 PDF 읽기

업로드한 PDF가 실험에 사용한 룰북과 같은지 내용으로 확인합니다. 각 페이지에서 본문을 읽고, 검색 결과에 표시할 PDF 쪽수를 붙입니다.


In [3]:
# 2. 룰북 PDF의 내용 확인 및 페이지별 텍스트 추출
from hashlib import sha256
from pathlib import Path

import pymupdf
from google.colab import files

rulebook_hash = "d00d1081a11d7f988f58769f0154aadcea0d32d8dbd04d31cef16980f57be985"

pdf_path = Path("Rules.pdf")
pdf_bytes = pdf_path.read_bytes() if pdf_path.is_file() else b""

if sha256(pdf_bytes).hexdigest() != rulebook_hash:
    uploaded = files.upload()
    for name, contents in uploaded.items():
        if sha256(contents).hexdigest() == rulebook_hash:
            pdf_path = Path(name)
            pdf_bytes = contents
            break
    else:
        raise ValueError("D&D Basic Rules v1.0 한국어판(179쪽) PDF를 업로드하세요. 노트북 맨 위에 실험에 사용한 룰북 링크가 있습니다.")

pages = []
with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:
    for page_no, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()
        if text:
            pages.append({"page": page_no, "text": text})

print("텍스트를 읽은 페이지:", len(pages))


Saving Rules.pdf to Rules.pdf
텍스트를 읽은 페이지: 179


## 문장 단위로 검색 데이터 만들기

PDF의 줄바꿈을 정리하고 문장·목록 경계에서 나눕니다. 다음 문장이 MiniLM 입력 한도 안에 들어오면 같은 조각에 넣고, 새 조각을 만들 때는 들어갈 공간이 있으면 앞 문장 하나를 겹칩니다. 한 문장이나 목록 항목이 너무 길면 검색용으로만 여러 번 나누고, 찾아온 원문은 항목 전체를 보여줍니다. 같은 모델과 검색용 원문으로 다시 실행할 때는 저장한 벡터를 재사용합니다.


In [4]:
# 3. 임베딩 모델과 토큰 한도 확인
import torch
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
model_revision = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
embedder = SentenceTransformer(model_name, revision=model_revision, device=embedding_device)
tokenizer = embedder.tokenizer
token_limit = embedder.max_seq_length
print(f"임베딩 장치: {embedding_device} | 입력 한도: {token_limit}토큰")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 장치: cuda | 입력 한도: 128토큰


In [5]:
# 4. 문장·목록 단위로 묶고 임베딩 생성
import re


def count_tokens(text):
    return len(tokenizer(text, truncation=False, verbose=False)["input_ids"])


def sentence_units(text):
    boundary = r"(?<=[.!?。！？])\s+|\n[ \t]*(?=[•●▪])|\n[ \t]*\n"
    return [" ".join(unit.split()) for unit in re.split(boundary, text) if unit.strip()]


def split_page(text):
    window = []
    for unit in sentence_units(text):
        if count_tokens(unit) > token_limit:
            if window:
                yield " ".join(window)
                window = []
            yield unit
            continue

        if window and count_tokens(" ".join(window + [unit])) > token_limit:
            yield " ".join(window)
            last_sentence = window[-1]
            window = [last_sentence] if count_tokens(f"{last_sentence} {unit}") <= token_limit else []
        window.append(unit)

    if window:
        yield " ".join(window)


def split_for_model(text, tokenizer, max_length, overlap_tokens=24):
    """긴 원문을 MiniLM이 읽을 수 있는 길이로 나눈다."""
    def length(s):
        return len(tokenizer(s, add_special_tokens=True, truncation=False, verbose=False)['input_ids'])
    if length(text) <= max_length:
        return [text]
    if not getattr(tokenizer, 'is_fast', False):
        raise ValueError('긴 조각 재분할에는 fast tokenizer가 필요합니다.')
    special = tokenizer.num_special_tokens_to_add(pair=False)
    budget = max_length - special
    if budget < 2:
        raise ValueError('모델 토큰 한도가 너무 작습니다.')
    offsets = tokenizer(text, add_special_tokens=False, truncation=False, verbose=False,
                        return_offsets_mapping=True)['offset_mapping']
    offsets = [(a, b) for a, b in offsets if b > a]
    pieces, start = [], 0
    while start < len(offsets):
        end = min(start + budget, len(offsets))
        # 잘라낸 경계에서 다시 토큰화하면 길이가 달라질 수 있으므로 재확인합니다.
        while end > start:
            left = 0 if start == 0 else offsets[start][0]
            right = len(text) if end == len(offsets) else offsets[end][0]
            part = text[left:right]
            if length(part) <= max_length:
                break
            end -= 1
        if end <= start:
            raise ValueError('한 토큰의 원문조차 모델 길이 안에 넣을 수 없습니다.')
        if part.strip():
            pieces.append(part)
        if end == len(offsets):
            break
        start = max(start + 1, end - min(overlap_tokens, budget // 4))
    assert pieces and all(length(p) <= max_length for p in pieces)
    return pieces


chunks = [
    {"page": page["page"], "text": part}
    for page in pages
    for part in split_page(page["text"])
]
texts = [chunk["text"] for chunk in chunks]

# 한도를 넘는 원문은 여러 임베딩에 연결하고 검색 결과에서는 한 번만 보여준다.
index_rows = [
    (position, part)
    for position, text in enumerate(texts)
    for part in split_for_model(text, tokenizer, token_limit)
]
index_texts = [part for _, part in index_rows]
max_tokens = max(map(count_tokens, index_texts))
assert max_tokens <= token_limit

# 모델과 입력이 같을 때, 이전에 만들어 둔 벡터를 재사용한다.
import json
from importlib.metadata import version
from tempfile import NamedTemporaryFile

import numpy as np

cache_key = sha256(json.dumps(
    [model_name, model_revision, embedding_device, str(torch.__version__),
     version("sentence-transformers"), version("transformers"), index_texts],
    ensure_ascii=False,
).encode("utf-8")).hexdigest()
cache_path = cache_dir / f"{cache_key}.npy"
expected_shape = (len(index_texts), embedder.get_sentence_embedding_dimension())

vectors = None
try:
    cached = np.load(cache_path, allow_pickle=False)
    if (cached.shape == expected_shape and np.issubdtype(cached.dtype, np.floating)
            and np.isfinite(cached).all()):
        vectors = cached
        print("이전에 저장한 벡터를 재사용합니다.")
    else:
        print("저장한 벡터의 형식이 달라 다시 만듭니다.")
except FileNotFoundError:
    pass
except (OSError, ValueError, EOFError):
    print("저장한 벡터를 읽을 수 없어 다시 만듭니다.")

if vectors is None:
    vectors = embedder.encode(index_texts, normalize_embeddings=True, show_progress_bar=True)
    temporary = None
    try:
        cache_dir.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(dir=cache_dir, suffix=".npy", delete=False) as output_file:
            temporary = Path(output_file.name)
            np.save(output_file, vectors, allow_pickle=False)
        temporary.replace(cache_path)
        print("이번에 만든 벡터를 저장했습니다.")
    except OSError as error:
        print(f"벡터 저장 실패. 이번 세션에서는 검색을 계속할 수 있습니다. ({error})")
        if temporary is not None:
            try:
                temporary.unlink(missing_ok=True)
            except OSError:
                pass

print(f"원문 조각: {len(chunks)}개 | 검색 벡터: {len(vectors)}개 | 최대 {max_tokens}/{token_limit}토큰")


/tmp/ipykernel_531/1785017635.py:101: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  expected_shape = (len(index_texts), embedder.get_sentence_embedding_dimension())


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

이번에 만든 벡터를 저장했습니다.
원문 조각: 2897개 | 검색 벡터: 2999개 | 최대 128/128토큰


In [6]:
# 5. 벡터와 원문 조각 번호를 Chroma에 저장
import chromadb
from uuid import uuid4

client = chromadb.EphemeralClient()
rules = client.create_collection(
    f"drowned_bell_rules_{uuid4().hex[:8]}",
    embedding_function=None,
    configuration={"hnsw": {"space": "cosine"}},
)
rules.add(
    ids=[f"part-{i}" for i in range(len(index_rows))],
    embeddings=vectors.tolist(),
    metadatas=[{"position": position} for position, _ in index_rows],
)
print("저장한 검색 벡터:", rules.count())


저장한 검색 벡터: 2999


## 검색 결과 확인

질문을 같은 임베딩 모델로 바꿔 가까운 청크 3개를 가져옵니다. 거리 값이 작을수록 가깝습니다. 출력된 쪽수와 원문을 보고 실제로 질문에 답할 근거인지 확인합니다.


In [7]:
# 6. 가까운 검색 벡터를 원문 조각으로 연결
def search_rules(question, top_k=3):
    query_vector = embedder.encode([question], normalize_embeddings=True).tolist()
    # 하나의 원문에 연결된 여러 벡터가 상위에 와도 필요한 원문 수를 확보한다.
    extra_vectors = len(index_rows) - len(chunks)
    matches = rules.query(
        query_embeddings=query_vector,
        n_results=min(top_k + extra_vectors, rules.count()),
        include=["metadatas", "distances"],
    )

    ranked = {}
    for meta, distance in zip(matches["metadatas"][0], matches["distances"][0]):
        ranked.setdefault(meta["position"], distance)
        if len(ranked) == top_k:
            break

    return {
        "ids": [[f"rule-{i}" for i in ranked]],
        "documents": [[chunks[i]["text"] for i in ranked]],
        "metadatas": [[{"file": pdf_path.name, "page": chunks[i]["page"]} for i in ranked]],
        "distances": [list(ranked.values())],
    }


def find_rules(question):
    result = search_rules(question)
    print(f"\n질문: {question}")
    for text, meta, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        print(f"\n{meta['file']} {meta['page']}쪽 | 거리 {distance:.3f}")
        print(text)


In [8]:
# 검색 결과와 출처를 확인하는 예시
find_rules("공격 굴림 결과가 상대 방어도와 같으면 명중해?")



질문: 공격 굴림 결과가 상대 방어도와 같으면 명중해?

Rules.pdf 160쪽 | 거리 0.219
기사는 근접 공격에 명중당했을 때 1회에 한해 AC에 +2 보 너스를 받아 그 공격을 빗나가게 할 수 있습니다. 이 반응행동을 사용하 려면 공격자를 볼 수 있어야 하고, 근접 무기를 장비하고 있어야 합니다. 기사는 국가의 지배자나 종교적 결사, 혹은 귀족적 이상에 맹세 한 전사들입니다. 기사의 성향에 따라 어떤 맹세를 어떻게 지키 고 있는가가 달라집니다.

Rules.pdf 75쪽 | 거리 0.234
만약 당신이 명중 굴림 을 굴려야 한다면, 그 행동은 공격으로 취급받는 것입니다. 명중 굴림 공격 행동을 하면, 명중 굴림의 결과에 따라 공격이 명중하거나 빗 나갑니다. 명중 굴림을 굴릴 때는 d20을 굴린 다음 해당하는 수정 치를 적용합니다. 만약 굴림의 결과가 목표의 방어도(AC) 이상이 라면 공격은 명중한 것입니다.

Rules.pdf 137쪽 | 거리 0.263
행동 물기. 근접 무기 공격: 명중 +0, 간격 5ft, 크리쳐 하나. 명중시: 1점의 관 통 피해.


다른 질문은 위 예시의 문장을 바꿔 검색할 수 있습니다.


## 키워드 검색 추가

의미 검색은 표현이 다른 문단도 찾고, [BM25](https://github.com/dorianbrown/rank_bm25)는 질문과 문단에 공통으로 나오는 표현을 활용합니다. 한국어는 어미가 달라도 일부가 겹칠 수 있도록 두 글자씩 나눕니다. 예를 들어 ‘붙잡힘’에서 ‘붙잡’과 ‘잡힘’을 만듭니다.

두 검색에서 후보를 각각 10개까지 가져옵니다. 높은 순위에 나온 후보에 더 큰 점수를 주고 두 목록의 점수를 더합니다. 이 방식은 [RRF](https://www.elastic.co/guide/en/elasticsearch/reference/current/rrf.html)이며, 합산 순위 상위 3개를 사용합니다.


In [9]:
import re
from rank_bm25 import BM25Okapi


def keyword_tokens(text):
    tokens = []
    for word in re.findall(r"[가-힣]+|[a-z0-9]+", text.casefold()):
        if "가" <= word[0] <= "힣":
            tokens.extend(word[i:i + 2] for i in range(len(word) - 1))
        else:
            tokens.append(word)
    return tokens


keyword_index = BM25Okapi([keyword_tokens(text) for text in texts])
chunk_positions = {f"rule-{i}": i for i in range(len(chunks))}


def search_hybrid(question):
    semantic_ids = search_rules(question, top_k=10)["ids"][0]
    scores = keyword_index.get_scores(keyword_tokens(question))
    keyword_positions = [i for i, score in enumerate(scores) if score > 0]
    keyword_positions.sort(key=lambda i: scores[i], reverse=True)
    keyword_ids = [f"rule-{i}" for i in keyword_positions[:10]]

    combined = {}
    for ranked_ids in (semantic_ids, keyword_ids):
        for rank, chunk_id in enumerate(ranked_ids, start=1):
            combined[chunk_id] = combined.get(chunk_id, 0) + 1 / (60 + rank)

    top_ids = sorted(combined, key=combined.get, reverse=True)[:3]
    selected = [chunks[chunk_positions[chunk_id]] for chunk_id in top_ids]
    return {
        "ids": [top_ids],
        "documents": [[chunk["text"] for chunk in selected]],
        "metadatas": [[{"file": pdf_path.name, "page": chunk["page"]} for chunk in selected]],
    }


## 검색 성능 비교

같은 10문항을 두 방식으로 검색합니다. 결과에서 룰북 쪽수와 질문에 답하는 기준 문구를 확인합니다.

- **첫 번째 결과에서 근거를 찾은 질문 수**를 셉니다.
- **첫 세 결과 안에서 근거를 찾은 질문 수**도 셉니다.

맞는 쪽이 나와도 찾아온 조각에 답이 되는 문구가 없으면 따로 표시합니다. 이 점수는 여기 있는 10문항의 검색 결과입니다.


In [10]:
# 질문, 근거 쪽수, 해당 근거에서 확인할 문구
evaluation_questions = [
    ("공격 굴림 결과가 상대 방어도와 같으면 명중해?", 75,
     "굴림의 결과가 목표의 방어도(AC) 이상이라면 공격은 명중한 것입니다."),
    ("짧은 휴식에서 회복 주사위를 어떻게 써?", 69,
     "HD 주사위 하나를 굴릴 때마다 여기에 자신의 건강 수정치를 더하며"),
    ("붙잡힌 상태에서 이동할 수 있어?", 170,
     "붙잡힌 상태의 크리쳐는 이동속도가 0이 됩니다."),
    ("이점이 있으면 두 주사위 중 어떤 걸 골라?", 59,
     "이점인 경우 둘 중 높은 값을 사용"),
    ("능력 판정에서 굴림 결과가 난이도와 같아도 성공이야?", 60,
     "결과값이 DC보다 높거나 같다면 판정은 성공한 것입니다."),
    ("적의 기회공격을 피하면서 물러나려면 어떤 행동을 써?", 74,
     "퇴각 행동을 사용하면, 당신의 이동은 그 턴이 끝날 때까지 기회공격을 유발하지 않습니다."),
    ("절반 엄폐는 방어도에 보너스를 얼마나 줘?", 76,
     "절반 엄폐를 받는 대상은 AC와 민첩 내성 굴림에 +2 보너스를 받습니다."),
    ("치명타가 나면 피해 주사위를 몇 번 굴려?", 77,
     "공격의 피해 주사위를 두 번 굴린 다음"),
    ("죽음 내성 굴림은 몇 이상이면 성공이야?", 78,
     "결과가 10 이상이라면, 당신은 성공한 것입니다."),
    ("1레벨 마법 화살을 쓰면 다트가 몇 개 나와?", 89,
     "빛나는 3개의 다트를 만들어 냅니다."),
]

In [11]:
def normalize_text(text):
    return "".join(text.split()).casefold()


def evaluate_search(search):
    first_hits = 0
    top3_hits = 0

    for question, expected_page, evidence in evaluation_questions:
        result = search(question)
        found_pages = [meta["page"] for meta in result["metadatas"][0]]
        expected_text = normalize_text(evidence)

        matched_rank = None
        for rank, (page, text) in enumerate(zip(found_pages, result["documents"][0]), 1):
            if page == expected_page and expected_text in normalize_text(text):
                matched_rank = rank
                break

        first_hits += matched_rank == 1
        top3_hits += matched_rank is not None
        rank_label = f"{matched_rank}번째 결과에서 근거를 찾음" if matched_rank else "근거를 찾지 못함"
        if matched_rank is None and expected_page in found_pages:
            rank_label = "맞는 쪽을 찾았지만 답이 되는 문장은 빠짐"
        print(f"{question} → {rank_label} (기준 {expected_page}쪽 / 검색 {found_pages})")

    total = len(evaluation_questions)
    print(f"첫 번째 결과에서 근거를 찾은 질문: {total}개 중 {first_hits}개")
    print(f"첫 세 결과 안에서 근거를 찾은 질문: {total}개 중 {top3_hits}개")


print("[의미 검색]")
evaluate_search(search_rules)

print("\n[의미 + 키워드 검색]")
evaluate_search(search_hybrid)


[의미 검색]
공격 굴림 결과가 상대 방어도와 같으면 명중해? → 2번째 결과에서 근거를 찾음 (기준 75쪽 / 검색 [160, 75, 137])
짧은 휴식에서 회복 주사위를 어떻게 써? → 맞는 쪽을 찾았지만 답이 되는 문장은 빠짐 (기준 69쪽 / 검색 [69, 69, 88])
붙잡힌 상태에서 이동할 수 있어? → 근거를 찾지 못함 (기준 170쪽 / 검색 [99, 99, 86])
이점이 있으면 두 주사위 중 어떤 걸 골라? → 근거를 찾지 못함 (기준 59쪽 / 검색 [32, 74, 19])
능력 판정에서 굴림 결과가 난이도와 같아도 성공이야? → 3번째 결과에서 근거를 찾음 (기준 60쪽 / 검색 [24, 95, 60])
적의 기회공격을 피하면서 물러나려면 어떤 행동을 써? → 맞는 쪽을 찾았지만 답이 되는 문장은 빠짐 (기준 74쪽 / 검색 [76, 76, 74])
절반 엄폐는 방어도에 보너스를 얼마나 줘? → 1번째 결과에서 근거를 찾음 (기준 76쪽 / 검색 [76, 32, 76])
치명타가 나면 피해 주사위를 몇 번 굴려? → 근거를 찾지 못함 (기준 77쪽 / 검색 [161, 103, 85])
죽음 내성 굴림은 몇 이상이면 성공이야? → 1번째 결과에서 근거를 찾음 (기준 78쪽 / 검색 [78, 93, 78])
1레벨 마법 화살을 쓰면 다트가 몇 개 나와? → 근거를 찾지 못함 (기준 89쪽 / 검색 [26, 56, 55])
첫 번째 결과에서 근거를 찾은 질문: 10개 중 2개
첫 세 결과 안에서 근거를 찾은 질문: 10개 중 4개

[의미 + 키워드 검색]
공격 굴림 결과가 상대 방어도와 같으면 명중해? → 1번째 결과에서 근거를 찾음 (기준 75쪽 / 검색 [75, 160, 75])
짧은 휴식에서 회복 주사위를 어떻게 써? → 3번째 결과에서 근거를 찾음 (기준 69쪽 / 검색 [69, 69, 69])
붙잡힌 상태에서 이동할 수 있어? → 맞는 쪽을 찾았지만 답이 되는 문장은 빠짐 (기준 170쪽 / 검색 [170, 76, 99])
이점이

## Qwen이 고른 근거 확인

먼저 검색 결과 세 개를 사용합니다. 그 결과에 나온 쪽에서 질문과 단어가 겹치는 다른 조각도 쪽마다 하나씩 살펴봅니다. 같은 쪽에 있더라도 첫 검색 결과에 필요한 문장이 빠질 수 있기 때문입니다.

Qwen은 질문에 직접 답하는 문장 번호를 고릅니다. 화면에는 그 문장의 원문과 쪽수를 표시합니다. **후보에 답이 없었는지**, **후보에 답은 있었지만 Qwen이 다른 문장을 골랐는지**를 구분해 확인합니다. 이 점검의 후보는 앞에서 점수를 낸 첫 세 결과보다 넓습니다.


In [12]:
# GPU 확인 후 토크나이저와 Qwen 불러오기
import torch

qwen_model = None
gpu_available = torch.cuda.is_available()

if gpu_available:
    from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

    print("GPU:", torch.cuda.get_device_name(0))
    qwen_id = "Qwen/Qwen3.5-4B"
    qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_id)
    qwen_dtype = (
        torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
        else torch.float16
    )
    qwen_model = Qwen3_5ForConditionalGeneration.from_pretrained(
        qwen_id, dtype=qwen_dtype, device_map="auto",
    )
    qwen_model.eval()
    print("Qwen 준비 완료")
else:
    print("CPU에서 실행 중입니다. 검색 평가 결과는 위에서 확인할 수 있습니다.")


GPU: Tesla T4


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Qwen 준비 완료


In [13]:
def evidence_candidates(question, result):
    # 검색에 나온 쪽에서 키워드가 겹치는 다른 조각을 하나씩 추가한다.
    positions = [chunk_positions[chunk_id] for chunk_id in result["ids"][0]]
    pages_to_check = dict.fromkeys(chunks[i]["page"] for i in positions)
    scores = keyword_index.get_scores(keyword_tokens(question))

    for page in pages_to_check:
        best = max(
            (i for i, chunk in enumerate(chunks)
             if chunk["page"] == page and i not in positions and scores[i] > 0),
            key=lambda i: scores[i],
            default=None,
        )
        if best is not None:
            positions.append(best)

    return list(dict.fromkeys(
        (chunks[i]["page"], sentence)
        for i in positions
        for sentence in sentence_units(chunks[i]["text"])
    ))


def select_evidence(question, candidates):
    if not candidates:
        return None

    numbered = "\n".join(
        f"{i}. [{page}쪽] {sentence}"
        for i, (page, sentence) in enumerate(candidates, start=1)
    )
    messages = [
        {
            "role": "system",
            "content": (
                "질문에 직접 답하는 룰북 문장 하나를 골라 숫자만 출력하세요. "
                "같은 주제를 언급해도 질문에 답하지 않으면 제외하세요. "
                "질문에서 행동하는 사람이 누구인지 확인하고, 문장에서 그 사람에게 적용되는 규칙만 고르세요. "
                "직접 답하는 문장이 없으면 0을 출력하세요."
            ),
        },
        {"role": "user", "content": f"질문: {question}\n\n룰북 후보:\n{numbered}"},
    ]
    inputs = qwen_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", enable_thinking=False,
    ).to(qwen_model.device)

    with torch.inference_mode():
        output = qwen_model.generate(**inputs, max_new_tokens=12, do_sample=False)

    input_length = inputs["input_ids"].shape[-1]
    answer = qwen_tokenizer.decode(output[0, input_length:], skip_special_tokens=True).strip()
    if re.fullmatch(r"[0-9]{1,3}", answer):
        index = int(answer)
        if 1 <= index <= len(candidates):
            return candidates[index - 1]
    return None


In [14]:
if qwen_model is not None:
    demo_questions = evaluation_questions[:3]
    selected_count = 0

    for question, expected_page, evidence in demo_questions:
        print(f"\n질문: {question}")
        result = search_hybrid(question)
        candidates = evidence_candidates(question, result)

        # 기준 문구는 채점에만 사용한다.
        expected_text = normalize_text(evidence)
        candidate_has_evidence = any(
            page == expected_page and expected_text in normalize_text(sentence)
            for page, sentence in candidates
        )
        selection = select_evidence(question, candidates)

        if selection is not None:
            page, sentence = selection
            print(f"선택한 룰북 원문 ({pdf_path.name} {page}쪽)")
            print(sentence)
            if page == expected_page and expected_text in normalize_text(sentence):
                selected_count += 1
                print("답이 되는 문장을 골랐습니다.")
                continue

        if not candidate_has_evidence:
            print("가져온 후보에 기준 문장이 없었습니다.")
        elif selection is None:
            print("후보에 답이 있었지만 Qwen이 문장을 고르지 못했습니다.")
        else:
            print("후보에 답이 있었지만 Qwen이 다른 문장을 골랐습니다.")

    print(f"\nQwen이 답이 되는 문장을 고른 질문: {len(demo_questions)}개 중 {selected_count}개")
else:
    print("Qwen 근거 선택은 GPU 런타임에서 실행합니다.")



질문: 공격 굴림 결과가 상대 방어도와 같으면 명중해?


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


선택한 룰북 원문 (Rules.pdf 75쪽)
만약 굴림의 결과가 목표의 방어도(AC) 이상이 라면 공격은 명중한 것입니다.
답이 되는 문장을 골랐습니다.

질문: 짧은 휴식에서 회복 주사위를 어떻게 써?
선택한 룰북 원문 (Rules.pdf 69쪽)
짧은 휴식을 취하고 나면, 캐릭터는 자신의 HD를 하나 혹은 그 이상 소비하여 회복을 시도할 수 있습니다.
후보에 답이 있었지만 Qwen이 다른 문장을 골랐습니다.

질문: 붙잡힌 상태에서 이동할 수 있어?
선택한 룰북 원문 (Rules.pdf 76쪽)
당신이 움직이게 되면, 당신은 붙잡 고 있는 크리쳐를 같이 움직일 수 있습니다.
후보에 답이 있었지만 Qwen이 다른 문장을 골랐습니다.

Qwen이 답이 되는 문장을 고른 질문: 3개 중 1개


검색 점수는 **처음 가져온 세 결과**를 기준으로 계산합니다. Qwen 근거 선택은 **같은 쪽의 다른 조각까지 살펴본 뒤** 고른 문장을 확인합니다. 두 결과는 각각 검색과 근거 선택을 보여줍니다.
